# Naive Bayes

In [5]:
import numpy as np
import pandas as pd

In [6]:
class GaussianNB:
    def __init__(self):
        self.classes = None
        self.mean = None
        self.var = None
        self.priors = None

    def fit(self, X, y, class_prior=None):
        self.classes = np.unique(y)
        n_samples, n_features = X.shape
        
        self.mean = np.zeros((len(self.classes), n_features))
        self.var = np.zeros((len(self.classes), n_features))
        self.priors = np.zeros(len(self.classes))
        
        for idx, c in enumerate(self.classes):
            X_c = X[y == c]
            self.mean[idx, :] = X_c.mean(axis=0)
            self.var[idx, :] = X_c.var(axis=0) + 1e-9
            
            if class_prior is not None and c in class_prior:
                self.priors[idx] = class_prior[c]
            else:
                self.priors[idx] = X_c.shape[0] / n_samples

        if class_prior is not None:
            self.priors = self.priors / np.sum(self.priors)

    def _pdf(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx]
        numerator = np.exp(- (x - mean) ** 2 / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator

    def predict_proba(self, X):
        n_samples = X.shape[0]
        log_probs = np.zeros((n_samples, len(self.classes)))
        
        for idx, c in enumerate(self.classes):
            prior = np.log(self.priors[idx])
            conditional = np.sum(np.log(self._pdf(idx, X) + 1e-9), axis=1)
            log_probs[:, idx] = prior + conditional
            
        log_probs -= np.max(log_probs, axis=1, keepdims=True)
        probs = np.exp(log_probs)
        probs /= np.sum(probs, axis=1, keepdims=True)
        
        pos_class_idx = np.where(self.classes == 1)[0][0]
        return probs[:, pos_class_idx]

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int)

In [7]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)
def precision(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0
def recall(y_true, y_pred):
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0
def f1(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


In [8]:
# Load test data 
test_df = pd.read_csv('data/data_encoded_test.csv')
X_test = test_df.drop(columns=['is_fraud']).values
y_test = test_df['is_fraud'].values

df = pd.read_csv('data/data_encoded_train.csv')
X_train = df.drop(columns=['is_fraud']).values
y_train = df['is_fraud'].values
gb = GaussianNB()
gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)

print("Final Metrics:")
print(f'accuracy: {accuracy(y_test,y_pred)}')
print(f'recall: {recall(y_test,y_pred)}')
print(f'f1 score: {f1(y_test,y_pred)}')
print(f'precision: {precision(y_test,y_pred)}')

Final Metrics:
accuracy: 0.972
recall: 0.6111111111111112
f1 score: 0.1358024691358025
precision: 0.0763888888888889
